# Filtro de derechos



**Escenario de aplicación** Filtro en tiempo real con la webcam: detecta la cara de la persona, estima su **género**, su **tono de piel** y si lleva el **pelo visible o cubierto**, y a partir de esos rasgos calcula un porcentaje de "derechos" según el país seleccionado.

**Teclas:** `e` medir · `n`/`b` cambiar país · `d` mostrar/ocultar parámetros (depuración) · `+`/`-` calibrar el offset del tono de piel · `Esc` salir.

*Se han utilizado estimaciones satíricas con objetivos humorísticos. No son mediciones reales.*

## **El filtro se realiza con tono humorístico sin objetivo de ofender a ningun colectivo**

En esta primera celda se importan las librerías: OpenCV para la cámara y el dibujo, DeepFace para el género, MediaPipe para segmentar el pelo y la piel, el `SelectorImagenes` propio para elegir el país y diferentes utilidades.

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import csv, time, ctypes, threading, unicodedata, urllib.request
from collections import deque, Counter

import cv2
import numpy as np
from deepface import DeepFace
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

from comun import Camara, DetectorCaras, texto
from selector import SelectorImagenes

## Parámetros ajustables

Aquí se se encuentran los diferentes parámetros para el uso del filtro:

- **Ficheros:** modelo de segmentación, CSV de países y carpeta de banderas.
- **Interfaz:** fuente, nombre de la ventana, resolución de la pantalla y se pueden mostrar los parametros seguidos para el calculo.
- **Métrica:** pesos de cada factor, que deben sumar 1.
- **Tono de piel y luz:** umbrales del ITA para clasificar en claro/intermedio/oscuro, histéresis y límites.
- **Pelo:** margen alrededor de la cara para recortar la cabeza y umbrales de color para clasificar el pelo.
- **Rendimiento:** cada cuánto se lanzan los análisis pesados.
- **Estado global:** colas de votos que suavizan las detecciones entre frames y flags para no lanzar dos hilos a la vez.

In [2]:
# Ficheros
SEG = "modelos/selfie_multiclass_256x256.tflite"
SEG_URL = "https://storage.googleapis.com/mediapipe-models/image_segmenter/selfie_multiclass_256x256/float32/latest/" + os.path.basename(SEG)
CSV_PAISES = "paises.csv"
CARPETA_BANDERAS = "banderas"

# Interfaz
FONT = cv2.FONT_HERSHEY_SIMPLEX
VENTANA = 'Filtro de derechos'
MOSTRAR_PARAMETROS = False
try:
    PW, PH = ctypes.windll.user32.GetSystemMetrics(0), ctypes.windll.user32.GetSystemMetrics(1)
except Exception:
    PW, PH = 1920, 1080
CLAVES = [("genero", "Genero"), ("tono", "Tono piel"), ("pelo", "Pelo"), ("color", "Color pelo")]

# Metrica
W_G, W_S, W_P = 0.4, 0.4, 0.2
assert abs(W_G + W_S + W_P - 1) < 1e-9
DURACION_MEDICION = 4.0

# Tono de piel y luz
UMBRAL_CLARO, UMBRAL_OSCURO, HISTERESIS = 41, 10, 4
MIN_PIXELES_PIEL = 300
LUZ_MIN, LUZ_MAX = 25, 92
SATURADOS_MAX = 0.05
ASIMETRIA_MAX = 18
ITA_OFFSET = 10

# Pelo
MARGEN_CABEZA = (0.8, 1.0, 0.8, 0.6)
PELO = {"L_negro": 28, "dL_negro": -38, "C_gris": 9, "C_fantasia": 28, "C_rojo": 24, "L_rubio": 55}
EXPOSICION = None

# Rendimiento
INTERVALO_PELO = 0.5
INTERVALO_PELO_LENTO = 1.5
INTERVALO_DF = 0.8
INTERVALO_DF_LENTO = 3.0
UMBRAL_MUJER = 0.30

# Banderas
ISO = {"espana": "es", "noruega": "no", "india": "in", "brasil": "br", "nigeria": "ng",
       "arabia saudi": "sa", "iran": "ir", "afganistan": "af"}

# Estados globales
cat_actual = "intermedio"
estado_luz = "..."
hist_ita = deque(maxlen=5)
votos_genero = deque(maxlen=5)
votos_color, votos_estado = deque(maxlen=8), deque(maxlen=5)
ocupado = {'df': False, 'pelo': False}
ultimo = {'df': 0.0, 'pelo': 0.0}

## Animación de la medición

La clase `Medicion` controla la animación que se produce cuando se pulsa la `E` y las diferentes variables que lo controlan.

In [3]:
class Medicion:
    def __init__(self):
        self.reset()
        self.aviso_hasta = 0.0

    def reset(self):
        self.estado, self.valor, self.objetivo, self.t0 = "espera", None, None, 0.0

    def iniciar(self, objetivo):
        self.estado, self.objetivo, self.t0 = "midiendo", objetivo, time.time()

    def actualizar(self, objetivo=None):
        if objetivo is not None:
            self.objetivo = objetivo              
        if self.estado != "midiendo":
            return
        t = time.time() - self.t0
        p = min(t / DURACION_MEDICION, 1.0)
        if p >= 1.0:
            self.estado, self.valor = "fijo", self.objetivo
            return
        ease = p * p * (3 - 2 * p)               
        amp = (1 - p) ** 1.3                      
        osc = 0.6 * np.sin(2 * np.pi * 1.1 * t) + 0.4 * np.sin(2 * np.pi * 2.3 * t + 1.0)
        base = 50 + (self.objetivo - 50) * ease
        self.valor = float(np.clip(base + 50 * amp * osc, 0, 100))

## Detector de caras DNN

Se usa el carga el detector de caras y le realiza ajuste para aumentar el tamaño de la cara para que pueda superar una confianza mínima.

- `suavizar_caja` mezcla la caja nueva con la anterior para que el recuadro no tiemble de un frame a otro.

In [4]:
detector = DetectorCaras(conf_min=0.5)

def suavizar_caja(a, b, alpha=0.5):
    return tuple(int(alpha * n + (1 - alpha) * o) for o, n in zip(a, b))

## Iluminación y tono de piel

Funciones para estimar el tono de piel de forma lo más robusta posible a la luz:

- `recortar_cabeza`: recorta la zona de la cabeza con margen para incluir el pelo.
- `ita_piel`: toma solo los píxeles de piel, los pasa a Lab y calcula el **ITA**. Además detecta si la luz es mala para no fiarse de esa medida.
- `categoria_estable`: convierte el ITA en claro / intermedio / oscuro usando histéresis, para estabilizar la categoría.

In [5]:
def recortar_cabeza(frame, caja):
    H, W = frame.shape[:2]
    x, y, w, h = caja
    ml, mt, mr, mb = MARGEN_CABEZA
    x0, y0 = max(int(x - ml * w), 0), max(int(y - mt * h), 0)
    x1, y1 = min(int(x + w + mr * w), W), min(int(y + h + mb * h), H)
    return frame[y0:y1, x0:x1], (x - x0, y - y0, w, h)

def ita_piel(crop, mask):
    piel = cv2.erode((mask == 3).astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)
    piel &= cv2.inRange(cv2.cvtColor(crop, cv2.COLOR_BGR2YCrCb), (0, 133, 77), (255, 173, 127)) > 0
    if piel.sum() < MIN_PIXELES_PIEL:
        return None, None, "sin piel"
    if (crop.max(axis=2)[piel] >= 250).mean() > SATURADOS_MAX:
        return None, None, "quemada"

    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).astype(np.float32)
    L, b = lab[..., 0] * 100 / 255, lab[..., 2] - 128
    cols = np.arange(piel.shape[1])[None, :]
    xm = np.median(np.nonzero(piel)[1])
    izq, der = piel & (cols < xm), piel & (cols >= xm)
    if izq.sum() < 50 or der.sum() < 50:
        return None, None, "sin piel"
    asim = abs(np.median(L[izq]) - np.median(L[der]))

    Lp, bp = L[piel], b[piel]
    lo, hi = np.percentile(Lp, [20, 80])
    k = (Lp >= lo) & (Lp <= hi)
    Lm, bm = float(np.median(Lp[k])), float(np.median(bp[k]))

    luz = ("oscura" if Lm < LUZ_MIN else "quemada" if Lm > LUZ_MAX
           else "lateral" if asim > ASIMETRIA_MAX else "ok")
    return float(np.degrees(np.arctan2(Lm - 50, bm))) + ITA_OFFSET, Lm, luz

def categoria_estable(ita):
    global cat_actual
    if cat_actual != "claro" and ita > UMBRAL_CLARO + HISTERESIS:
        cat_actual = "claro"
    elif cat_actual != "oscuro" and ita < UMBRAL_OSCURO - HISTERESIS:
        cat_actual = "oscuro"
    elif cat_actual == "claro" and ita < UMBRAL_CLARO - HISTERESIS:
        cat_actual = "intermedio"
    elif cat_actual == "oscuro" and ita > UMBRAL_OSCURO + HISTERESIS:
        cat_actual = "intermedio"
    return cat_actual

## Pelo (MediaPipe Image Segmenter)

- `segmentar`: devuelve la máscara de clases para una imagen.
- `clasificar_color` / `color_pelo`: calculan el color medio del pelo en Lab y lo clasifican en negro, castaño, rubio, pelirrojo, canoso o fantasía.
- `estado_pelo`: mira la zona de encima de la frente y los laterales para decidir si el pelo está visible, parcialmente cubierto, cubierto o si no hay pelo.

Al final se hace una segmentación de prueba para acelerar la primera ejecución.

In [6]:
if not os.path.exists(SEG):
    urllib.request.urlretrieve(SEG_URL, SEG)

opts = vision.ImageSegmenterOptions(
    base_options=mp_python.BaseOptions(model_asset_path=SEG),
    output_category_mask=True,
    running_mode=vision.RunningMode.IMAGE)
segmenter = vision.ImageSegmenter.create_from_options(opts)

def segmentar(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    m = np.squeeze(segmenter.segment(img).category_mask.numpy_view().copy())
    if m.shape != frame.shape[:2]:
        m = cv2.resize(m.astype(np.uint8), (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST)
    return m.astype(np.uint8)

def clasificar_color(L, C, H, dL):
    P = PELO
    if C > P["C_fantasia"] and not (15 <= H <= 105):
        return "fantasia"                                
    if L < P["L_negro"] or (dL is not None and dL < P["dL_negro"]):
        return "negro"
    if C < P["C_gris"]:
        return "canoso" if L > 45 else "negro"
    if C > P["C_rojo"] and H < 55:
        return "pelirrojo"
    return "rubio" if L > P["L_rubio"] else "castano"

def color_pelo(crop, mask, L_piel):
    pelo = cv2.erode((mask == 1).astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)
    if pelo.sum() < 150:
        return None
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).astype(np.float32)[pelo]
    L, a, b = lab[:, 0] * 100 / 255, lab[:, 1] - 128, lab[:, 2] - 128
    lo, hi = np.percentile(L, [15, 85])
    k = (L >= lo) & (L <= hi)
    L, a, b = np.median(L[k]), np.median(a[k]), np.median(b[k])
    C, H = np.hypot(a, b), np.degrees(np.arctan2(b, a)) % 360
    return clasificar_color(L, C, H, None if L_piel is None else L - L_piel)

def estado_pelo(mask, caja_rel):
    """Corona (sobre la frente) y laterales: distingue visible / parcial / cubierto / sin pelo."""
    H, W = mask.shape
    x, y, w, h = caja_rel
    def zona(x0, y0, x1, y1):
        z = mask[max(int(y0), 0):min(int(y1), H), max(int(x0), 0):min(int(x1), W)]
        return z if z.size else None
    corona = zona(x, y - 0.5 * h, x + w, y + 0.1 * h)
    if corona is None:
        return None
    lados = [z for z in (zona(x - 0.35 * w, y, x, y + 0.7 * h),
                         zona(x + w, y, x + 1.35 * w, y + 0.7 * h)) if z is not None]
    pelo_c = (corona == 1).mean()
    tapa_c = np.isin(corona, (4, 5)).mean()                 
    pelo_l = np.mean([(z == 1).mean() for z in lados]) if lados else 0.0
    if tapa_c > 0.35:
        return "cubierto" if pelo_l < 0.10 else "parcial"  
    if pelo_c > 0.35:
        return "visible"
    if pelo_c > 0.10 or pelo_l > 0.15:
        return "parcial"
    return "sin pelo"

segmentar(np.zeros((256, 256, 3), np.uint8))

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(256, 256), dtype=uint8)

## Género (DeepFace) + votos y workers en hilos

- `moda`, `normalizar_genero`: utilidades para resumir los votos.
- `cara_con_margen`: recorta la cara con algo de margen, que es lo que espera DeepFace.
- `worker_df`: analiza el género con DeepFace en un hilo.
- `worker_pelo`: en otro hilo segmenta la cabeza, calcula el ITA de la piel y el estado de la luz, y vota el estado del pelo y su color.
- `lanzar_workers`: decide en cada frame si toca lanzar un nuevo análisis de género o de pelo, respetando los intervalos y sin lanzar dos a la vez.

Al final se hace una llamada de prueba a DeepFace para cargar el modelo antes de abrir la cámara.

In [7]:
def moda(d, defecto=None):
    l = list(d)
    return Counter(l).most_common(1)[0][0] if l else defecto

def normalizar_genero(g):
    g = str(g).lower()
    if g in ("woman", "female", "mujer"): return "m"
    if g in ("man", "male", "hombre"):    return "h"
    return None

def cara_con_margen(frame, caja, m=0.25):
    H, W = frame.shape[:2]
    x, y, w, h = caja
    dx, dy = int(m * w), int(m * h)
    return frame[max(y - dy, 0):min(y + h + dy, H), max(x - dx, 0):min(x + w + dx, W)]

def worker_df(cara):
    try:
        r = DeepFace.analyze(cara, actions=['gender'], detector_backend='skip',
                             enforce_detection=False, silent=True)[0]
        votos_genero.append(float(r['gender']['Woman']) / 100)
    except Exception as e:
        print("Error genero:", e)
    finally:
        ocupado['df'] = False

def worker_pelo(crop, caja_rel):
    global estado_luz
    try:
        mask = segmentar(crop)
        ita, L_piel, luz = ita_piel(crop, mask)
        estado_luz = luz
        if ita is not None and luz == "ok":          # solo se fia del ITA con buena luz
            hist_ita.append(ita)
        estado = estado_pelo(mask, caja_rel)
        if estado:
            votos_estado.append(estado)
        if estado in ("visible", "parcial"):         # sin pelo a la vista no hay color
            color = color_pelo(crop, mask, L_piel)
            if color:
                votos_color.append(color)
    except Exception as e:
        print("Error pelo/piel:", e)
    finally:
        ocupado['pelo'] = False

def lanzar_workers(frame, caja):
    ahora = time.time()
    intervalo_df = INTERVALO_DF if len(votos_genero) < 3 else INTERVALO_DF_LENTO
    intervalo_pelo = INTERVALO_PELO if len(hist_ita) < 3 else INTERVALO_PELO_LENTO
    if not ocupado['df'] and ahora - ultimo['df'] > intervalo_df:
        ocupado['df'] = True; ultimo['df'] = ahora
        threading.Thread(target=worker_df, args=(cara_con_margen(frame, caja).copy(),), daemon=True).start()
    if not ocupado['pelo'] and ahora - ultimo['pelo'] > intervalo_pelo:
        ocupado['pelo'] = True; ultimo['pelo'] = ahora
        crop, caja_rel = recortar_cabeza(frame, caja)
        threading.Thread(target=worker_pelo, args=(crop.copy(), caja_rel), daemon=True).start()

DeepFace.analyze(np.zeros((224, 224, 3), np.uint8), actions=['gender'],
        detector_backend='skip', enforce_detection=False, silent=True)

26-11-25 18:13:40 - 🔗 gender_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/gender_model_weights.h5 to C:\Users\vidal\.deepface\weights\gender_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/gender_model_weights.h5
To: C:\Users\vidal\.deepface\weights\gender_model_weights.h5
100%|██████████| 537M/537M [00:06<00:00, 85.3MB/s] 


[{'gender': {'Woman': np.float32(49.649067), 'Man': np.float32(50.35093)},
  'dominant_gender': 'Man',
  'region': {'x': 0,
   'y': 0,
   'w': 224,
   'h': 224,
   'left_eye': None,
   'right_eye': None},
  'face_confidence': 0.0}]

## Datos de países y métrica

Lee `paises.csv` y guarda para cada país los factores por género, el tono mayoritario, el factor para grupos minoritarios y la regla sobre el pelo.

- `calcular_derechos`: combina los tres factores con los pesos `W_G`, `W_S`, `W_P` y devuelve el porcentaje total y el desglose.

In [8]:
def num(s):
    return float(str(s).strip().replace(",", "."))

paises = []
with open(CSV_PAISES, encoding="utf-8-sig", newline="") as f:
    primera = f.readline()
    f.seek(0)
    delim = ";" if primera.count(";") >= primera.count(",") else ","
    for r in csv.DictReader(f, delimiter=delim):
        if not (r.get("pais") or "").strip():
            continue
        r = {k.strip(): (v or "").strip() for k, v in r.items() if k}
        for k in ("f_genero_m", "f_genero_h", "f_grupo"):
            r[k] = num(r[k])
        r["tono_mayoritario"] = r["tono_mayoritario"].lower()
        r["regla_pelo"] = r["regla_pelo"].lower()
        paises.append(r)

def calcular_derechos(p, genero, tono, pelo):
    f_g = p["f_genero_m"] if genero == "m" else p["f_genero_h"]
    f_s = 1.0 if tono == p["tono_mayoritario"] else p["f_grupo"]
    f_p = {"visible": 0.3, "parcial": 0.6}.get(pelo, 1.0) if p["regla_pelo"] == "cubrir" else 1.0
    total = 100 * (W_G * f_g + W_S * f_s + W_P * f_p)
    return total, f_g, f_s, f_p

## Selector de país con banderas

Genera la lista de países para el selector: a cada país le asigna los datos del csv y la URL de su bandera en flagcdn. `SelectorImagenes` descarga y guarda las banderas en caché y las muestra en la parte de arriba de la imagen.

In [9]:
def norm(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower().strip()

def iso_de(p):
    return (p.get("iso") or ISO.get(norm(p["pais"])) or "").lower()

items = [{"nombre": p["pais"],
          "imagen": f"https://flagcdn.com/w160/{iso_de(p)}.png" if iso_de(p) else None,
          "etiqueta": iso_de(p)} for p in paises]

selector = SelectorImagenes(items, carpeta_cache=CARPETA_BANDERAS, zona_sup=20)

Imagenes cargadas: 8 de 8


## Interfaz: panel de parámetros

`color_total` elige el color de la barra según el porcentaje.

`dibujar_panel` dibuja a la izquierda un panel semitransparente con lo que se ha detectado de la persona. **Está oculto por defecto**; se muestra con la tecla `d` para comprobar qué está midiendo el programa.

In [10]:
def color_total(t):
    return (0, 200, 0) if t >= 66 else (0, 200, 255) if t >= 33 else (0, 0, 230)

def dibujar_panel(frame, det, ita_m):
    x0, y0, w, h = 8, 40, 215, 34 + 26 * len(CLAVES)
    over = frame.copy()
    cv2.rectangle(over, (x0, y0), (x0 + w, y0 + h), (20, 20, 20), -1)
    cv2.addWeighted(over, 0.65, frame, 0.35, 0, frame)
    cv2.putText(frame, "PARAMETROS", (x0 + 8, y0 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    for k, (key, nombre) in enumerate(CLAVES):
        y = y0 + 48 + 26 * k
        v = det[key] if det[key] else "..."
        extra = f" ({ita_m:.0f})" if key == "tono" and ita_m is not None else ""
        cv2.putText(frame, f"{nombre}: {v}{extra}", (x0 + 8, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.47, (255, 255, 255), 1)

## Funciones para el bucle

Funciones auxiliares que usa el bucle principal:

- `txt`: escribe texto con borde negro para que se lea sobre cualquier fondo.
- `a_pantalla`: escala el frame a la resolución de la pantalla manteniendo la proporción.
- `widget_cabeza`: dibuja sobre la cabeza la pregunta y la barra con el porcentaje o el mensaje de estado.
- `objetivo`: calcula el porcentaje final con los datos detectados y el país elegido.
- `reiniciar_sujeto` / `cambio_persona`: si cambia la persona delante de la cámara o desaparece la cara, se borran todos los votos y se empieza de cero.
- `tecla`: gestiona las teclas.

In [11]:
def txt(img, s, x, y, tam=0.55):
    texto(img, s, x, y, tam)

lienzo_pant = np.zeros((PH, PW, 3), np.uint8)

def a_pantalla(frame):
    h, w = frame.shape[:2]
    s = min(PW / w, PH / h)
    nw, nh = int(w * s), int(h * s)
    y0, x0 = (PH - nh) // 2, (PW - nw) // 2
    lienzo_pant[y0:y0 + nh, x0:x0 + nw] = cv2.resize(frame, (nw, nh))
    return lienzo_pant

def widget_cabeza(frame, caja, med, pais, listo):
    H, W = frame.shape[:2]
    x, y, w, h = caja
    pregunta = f"Cuantos derechos tienes en {pais}?"
    tw = cv2.getTextSize(pregunta, FONT, 0.55, 2)[0][0]
    ancho = max(w, tw + 10, 220)
    bx = int(np.clip(x + w // 2 - ancho // 2, 4, W - ancho - 4))
    by = max(y - 40, 34)
    bh = 28

    txt(frame, pregunta, bx + (ancho - tw) // 2, by - 8)
    cv2.rectangle(frame, (bx, by), (bx + ancho, by + bh), (60, 60, 60), -1)
    if time.time() < med.aviso_hasta:
        mensaje = "Aun analizando..."
    elif med.valor is not None:
        cv2.rectangle(frame, (bx, by), (bx + int(ancho * med.valor / 100), by + bh), color_total(med.valor), -1)
        mensaje = f"{med.valor:.0f}%" + ("  (E: medir de nuevo)" if med.estado == "fijo" else "")
    else:
        mensaje = ("Pulsa E para averiguarlo" if listo else
                   f"Luz: {estado_luz}" if estado_luz not in ("ok", "...") else "Analizando...")
    txt(frame, mensaje, bx + 8, by + 20)


def objetivo():
    return calcular_derechos(paises[selector.i], val["genero"], val["tono"], val["pelo"])[0] if val else None

def reiniciar_sujeto():
    global cat_actual, estado_luz, ita_m
    for d in (hist_ita, votos_genero, votos_color, votos_estado):
        d.clear()
    cat_actual, estado_luz, ita_m = "intermedio", "...", None
    ultimo['df'] = ultimo['pelo'] = 0.0
    med.reset()

def cambio_persona(a, b):
    ix = max(0, min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / (a[2] * a[3] + b[2] * b[3] - inter + 1e-9) < 0.3

def tecla(k, frame):
    global full, MOSTRAR_PARAMETROS, ITA_OFFSET, cat_actual
    if k == ord('e'):
        o = objetivo()
        if o is None:
            med.aviso_hasta = time.time() + 2.0
        elif med.estado != "midiendo":
            med.iniciar(o)
    elif k in (ord('n'), ord('b')):
        selector.siguiente() if k == ord('n') else selector.anterior()
        med.reset()
    elif k == ord('f'):
        full = not full
        cv2.setWindowProperty(VENTANA, cv2.WND_PROP_FULLSCREEN,
                              cv2.WINDOW_FULLSCREEN if full else cv2.WINDOW_NORMAL)
    elif k == ord('d'):
        MOSTRAR_PARAMETROS = not MOSTRAR_PARAMETROS
    elif k in (ord('+'), ord('-')):
        ITA_OFFSET += 3 if k == ord('+') else -3
        hist_ita.clear(); cat_actual = "intermedio"
        print("ITA_OFFSET =", ITA_OFFSET)

## Bucle principal

Abre la cámara y la ventana en pantalla completa y, en cada frame nuevo:

1. Detecta la cara y suaviza su posición; si cambia de persona o se pierde la cara, reinicia los datos.
2. Lanza en segundo plano los análisis de género y de pelo/piel.
3. Combina los votos acumulados en las detecciones finales.
4. Actualiza la animación de medida, el selector de país y el widget sobre la cabeza.

In [ ]:
cam = Camara(exposicion=EXPOSICION)
cv2.namedWindow(VENTANA, cv2.WINDOW_NORMAL)
cv2.setWindowProperty(VENTANA, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
full = True
ita_m, val, caja = None, None, None
last_id, i, perdidos, t_sin_cara = -1, 0, 0, None
med = Medicion()
try:
    while True:
        fid, frame = cam.read()
        if frame is None or fid == last_id:
            if cv2.waitKey(1) & 0xFF == 27:
                break
            continue
        last_id, i = fid, i + 1

        if caja is None or i % 3 == 0:
            nueva = detector.mayor(frame)
            if nueva:
                if caja is not None and cambio_persona(caja, nueva):
                    reiniciar_sujeto()
                caja = nueva if caja is None else suavizar_caja(caja, nueva)
                perdidos, t_sin_cara = 0, None
            else:
                perdidos += 1
                if perdidos > 4:
                    caja = None
                    t_sin_cara = t_sin_cara or time.time()
        if caja:
            lanzar_workers(frame, caja)
            ita_m = float(np.median(hist_ita)) if hist_ita else None
            p_m = float(np.mean(votos_genero)) if votos_genero else None
            genero = None if p_m is None else ("m" if p_m > UMBRAL_MUJER else "h")
            det = {"genero": genero,
                   "tono": categoria_estable(ita_m) if ita_m is not None else None,
                   "pelo": moda(votos_estado),
                   "color": moda(votos_color)}
            val = det if (det["genero"] and det["tono"] and det["pelo"]) else None
            if MOSTRAR_PARAMETROS:
                dibujar_panel(frame, det, ita_m)
        else:
            val = None
            txt(frame, "no detectado", 10, 30, 0.8)
            if t_sin_cara and time.time() - t_sin_cara > 0.4:
                reiniciar_sujeto()

        med.actualizar(objetivo())
        selector.dibujar(frame)
        if caja:
            widget_cabeza(frame, caja, med, selector.nombre, val is not None)
            texto(frame, "n/b partido  f pantalla  Esc salir",
              10, frame.shape[0] - 12, 0.45)

        cv2.imshow(VENTANA, a_pantalla(frame) if full else frame)
        k = cv2.waitKey(1) & 0xFF
        if k == 27:
            break
        tecla(k, frame)
finally:
    cam.release()
    cv2.destroyAllWindows()